Começando a explorar a tabela orders e suas propriedades utlizando da biblioteca padrão do python csv.

Quero primeiro ler o arquivo csv e entender a estrutura dos dados, como colunas, tipos de dados e possíveis valores nulos. Em seguida, vou converter para um arquivo .db para armazenar esses dados e realizar consultas SQL para analisar as informações contidas no arquivo utilizando sqlite3 do python ou utilizando o datagrip/dbeaver.

In [ ]:
"Importando biblioteca padrão do python para manipulação de arquivos csv e do sqlite3 para manipulação de banco de dados."

import csv

# 1. definindo o caminho do arquivo de origem orders.csv
CSV_PATH = r"..\..\lh_nautical_csv\orders.csv"


# 2. criando uma função para verificar o tipo de cada valor por coluna
def inferir_tipo(valor):
    """Função para inferir o tipo de dado de um valor de uma coluna."""
    if valor is None or valor == "":
        return "vazio"  # se o valor for nulo ou vazio, retorna "vazio"
    try:
        int(valor)
        return "inteiro"  # se o valor for um inteiro, retorna "inteiro"
    except ValueError:  # se o valor não for um inteiro, ignora e continua
        pass
    try:
        float(valor)
        return "decimal"  # se o valor for um decimal, retorna "decimal"
    except ValueError:  # se o valor não for um decimal, ignora e continua
        pass
    if valor.lower() in ("true", "false"):
        return "booleano"  # se o valor for um booleano, retorna "booleano"
    return "texto"  # se o valor não for nenhum dos tipos acima, retorna "texto"


# 3. criando uma função para formatar o valor de cada coluna
# para deixar num visual de tabela no print
def formatar_valor(valor, limite=30):
    """Diminui o texto e adiciona reticências caso ultrapasse o limite de 30 caracteres."""
    if valor is None or valor == "":
        return "<vazio>"  # se o valor for nulo ou vazio, retorna "<vazio>"
    texto = str(valor)
    if len(texto) > limite:
        return (
            texto[: limite - 3] + "..."
        )  # se o valor ultrapassar o limite, retorna os primeiros 27 caracteres + "..."
    return texto


# 4. recebendo o nome das colunas do arquivo csv e o primeiro valor
# não vazio de cada coluna por meio das funções inferir_tipo e formatar_valor
with open(CSV_PATH, mode="r", encoding="utf-8") as f:
    reader = csv.DictReader(f)  # lendo o arquivo csv como um dicionário
    colunas = reader.fieldnames or []  # obtendo o nome das colunas do arquivo csv

    if not colunas:  # Printa se o arquivo csv está vazio ou não possui cabeçalho
        print("O arquivo CSV está vazio ou não possui cabeçalho.")
    else:
        primeiros_valores: dict[str, str | None] = {
            coluna: None for coluna in colunas
        }  # inicializando um dicionário para armazenar o primeiro valor não vazio de cada coluna

        for linha in reader:
            for coluna in colunas:
                if primeiros_valores[coluna] is None:
                    valor_linha = linha[coluna]
                    if valor_linha not in (None, ""):
                        primeiros_valores[coluna] = valor_linha
                        # se o valor da coluna for diferente de None ou
                        # vazio, armazena o valor no dicionário primeiros_valores
        # imprimindo o cabeçalho da tabela e o resumo das colunas, valores de exemplo e tipos de dados
        print("\nResumo da tabela orders\n")
        print("Quantidade de colunas:", len(colunas))
        print(f"{'Coluna':<25} | {'Valor de exemplo':<30} | Tipo")
        print("-" * 72)

        for coluna in colunas:
            valor_exemplo = primeiros_valores[coluna]
            print(
                f"{coluna:<25} | {formatar_valor(valor_exemplo):<30} | {inferir_tipo(valor_exemplo)}"
            )  # imprimindo o nome da coluna, o valor de exemplo e o tipo de dado


Resumo da tabela orders

Quantidade de colunas: 13
Coluna                    | Valor de exemplo               | Tipo
------------------------------------------------------------------------
id                        | 1                              | inteiro
order_number              | SO-000001                      | texto
channel                   | ecommerce                      | texto
customer_id               | 1136                           | inteiro
salesperson_id            | 9                              | inteiro
location_id               | 1                              | inteiro
status                    | paid                           | texto
subtotal                  | 323.34                         | decimal
discount_amount           | 35.57                          | decimal
total                     | 287.77                         | decimal
placed_at                 | 2022-09-06 05:37:37            | texto
created_at                | 2022-09-06 05:37:37           

Após primeira análise do arquivo csv, percebi que ela possui 13 colunas, sendo elas:
- id de valor do tipo inteiro e chave primária
- order_number de valor do tipo texto
- channel de valor do tipo texto
- customer_id de valor do tipo inteiro
- salesperson_id de valor do tipo real (real pois contém valores nulos no e-commerce)
- location_id de valor do tipo inteiro
- status de valor do tipo texto
- subtotal de valor do tipo Real (real pois contém valores decimais)
- discount_amount de valor do tipo Real (real pois contém valores decimais)
- total de valor do tipo Real (real pois contém valores decimais)
- placed_at de valor do tipo texto
- created_at de valor do tipo texto
- updated_at de valor do tipo texto

Com essas informações, posso prosseguir para a próxima etapa que é a conversão do arquivo csv para um banco de dados sqlite e realizar consultas SQL para analisar os dados contidos na tabela orders.

In [ ]:
import sqlite3

# 1. definindo o caminho do arquivo de destino orders.sqlite
DB_PATH = sqlite3.connect("orders.sqlite")